In [ ]:
# Get youtube search results in sorted order
# https://www.thepythoncode.com/article/using-youtube-api-in-python

# => pip3 install --upgrade google-api-python-client google-auth-httplib2 google-auth-oauthlib

In [ ]:
from googleapiclient.discovery import build
from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.transport.requests import Request

import urllib.parse as p
import re
import os
import pickle
import pandas as pd
import json
from datetime import date
import glob
import pprint

SCOPES = ["https://www.googleapis.com/auth/youtube.force-ssl"]

In [ ]:
WORKDIR = os.getcwd()
print(WORKDIR)

# data retrieval section

In [ ]:
def youtube_authenticate():
    os.environ["OAUTHLIB_INSECURE_TRANSPORT"] = "1"
    api_service_name = "youtube"
    api_version = "v3"
    client_secrets_file = "client_secret_147818272428-2usk09rcbmjajo29gcva3pvp9csj53u9.apps.googleusercontent.com.json"
    creds = None
    # the file token.pickle stores the user's access and refresh tokens, and is
    # created automatically when the authorization flow completes for the first time
    if os.path.exists("token.pickle"):
        with open("token.pickle", "rb") as token:
            creds = pickle.load(token)
    # if there are no (valid) credentials availablle, let the user log in.
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file(client_secrets_file, SCOPES)
            creds = flow.run_local_server(port=0)
        # save the credentials for the next run
        with open("token.pickle", "wb") as token:
            pickle.dump(creds, token)

    return build(api_service_name, api_version, credentials=creds)

# authenticate to YouTube API
youtube = youtube_authenticate()

In [ ]:
def get_video_id_by_url(url):
    """
    Return the Video ID from the video `url`
    """
    # split URL parts
    parsed_url = p.urlparse(url)
    # get the video ID by parsing the query of the URL
    video_id = p.parse_qs(parsed_url.query).get("v")
    if video_id:
        return video_id[0]
    else:
        raise Exception(f"Wasn't able to parse video URL: {url}")

In [ ]:
#The below function gets a YouTube service object (returned from youtube_authenticate() function), as well as any keyword argument accepted by the API, and returns the API response for a specific video:

def get_video_details(youtube, **kwargs):
    return youtube.videos().list(
        part="snippet,contentDetails,statistics",
        **kwargs
    ).execute()

In [ ]:
#We also pass kwargs to the API directly. Next, let's define a function that takes a response returned from the above get_video_details() function, and prints the most useful information from a video:

def print_video_infos(video_response):
    items = video_response.get("items")[0]
    # get the snippet, statistics & content details from the video response
    snippet         = items["snippet"]
    statistics      = items["statistics"]
    content_details = items["contentDetails"]
    # get infos from the snippet
    channel_title = snippet["channelTitle"]
    title         = snippet["title"]
    description   = snippet["description"]
    publish_time  = snippet["publishedAt"]
    # get stats infos
    comment_count = statistics["commentCount"]
    like_count    = statistics["likeCount"]
    #DEPRECATED dislike_count = statistics["dislikeCount"]
    view_count    = statistics["viewCount"]
    # get duration from content details
    duration = content_details["duration"]
    # duration in the form of something like 'PT5H50M15S'
    # parsing it to be something like '5:50:15'
    parsed_duration = re.search(f"PT(\d+H)?(\d+M)?(\d+S)", duration).groups()
    duration_str = ""
    for d in parsed_duration:
        if d:
            duration_str += f"{d[:-1]}:"
    duration_str = duration_str.strip(":")
    print(f"""\
    Title: {title}
    Description: {description}
    Channel Title: {channel_title}
    Publish time: {publish_time}
    Duration: {duration_str}
    Number of comments: {comment_count}
    Number of likes: {like_count}
    Number of views: {view_count}
    """)
    #DEPRECATED Number of dislikes: {dislike_count}

video_url = "https://www.youtube.com/watch?v=jNQXAC9IVRw&ab_channel=jawed"
### parse video ID from URL
video_id = get_video_id_by_url(video_url)
### make API call to get video info
response = get_video_details(youtube, id=video_id)
### print extracted video infos
print_video_infos(response)

In [ ]:
def search(youtube, **kwargs):
    return youtube.search().list(
        part="snippet",
        **kwargs
    ).execute()
#You can also specify the order parameter in search() function to order search results, which can be 'date', 'rating', 'viewCount', 'relevance' (default), 'title', and 'videoCount'.
#Another useful parameter is the type, which can be 'channel', 'playlist' or 'video', default is all of them.
#Please check this page for more information about the search().list() method.
# https://developers.google.com/youtube/v3/docs/search/list

In [ ]:
def getsearchresultsbykeyword(keyword, amount, orderedby, lang):
    #order="relevance"or"viewCount"
    collected_responses = {}
    count = 0
    pT = ""
    while count < amount:
        try:
            response = search(youtube, q=keyword, maxResults=50, order=orderedby, type="video", pageToken=pT, relevanceLanguage=lang)
            #print(response)
            items = response.get("items")
            print(len(items))
            #print(items)
            count += len(items)
            print(count)
            print('adding items')
            if len(items)>0:
                for item in items:
                    #print(item)
                    collected_responses[len(collected_responses)] = item
                if len(items) >= 50:
                    pT=response['nextPageToken']
                else:
                    break
        except:
            count = amount
    return collected_responses

# data processing section

### only used one time to get up to date with the previous datasets

In [ ]:
def findalljsonfilesindir(dir):
    os.chdir(WORKDIR)
#    print(os.listdir())
    result = []
    for file in os.listdir(dir):
        if file.endswith('.json') and not file.startswith('client'):
    #        print(os.path.join(dir, file))
            result.append(os.path.join(dir, file))
    return result
#jsonfilelist = findalljsonfilesindir(paths['StreetFighter'])
#print(jsonfilelist)

In [ ]:
def tmpdef(name, filename):
    complist = pd.read_csv(os.path.join(paths[name], f'{name}AllIdTable.csv'))
    #print(complist.head())
    with open(filename, 'r') as file:
        data = json.load(file)
    if data:
        table = pd.DataFrame(columns = ['videoId', 'publishedAt', 'channelId', 'title',
                                        'description', 'channelTitle', 'publishTime', 'previousappearances'])
        for key in data:
            print(key)
            print(data[key]['id']['videoId'])
            #matches = complist[complist.eq(data[key]['id']['videoId']).any(axis=1)]
            matches = complist[complist.isin([data[key]['id']['videoId']])].stack()
            matches.index = matches.index.droplevel()
            matches = [x for x in matches.index if x.split('_')[-1] != '2023-10-16']
            print(matches)
            tmp = {'videoId': [data[key]['id']['videoId']],
                   'publishedAt': [data[key]['snippet']['publishedAt']],
                   'channelId': [data[key]['snippet']['channelId']],
                   'title': [data[key]['snippet']['title']],
                   'description': [data[key]['snippet']['description']],
                   'channelTitle': [data[key]['snippet']['channelTitle']],
                   'publishTime': [data[key]['snippet']['publishTime']],
                   'previousappearances': str(matches)}
            table = pd.concat([table, pd.DataFrame.from_dict(tmp)], ignore_index = True)
    else:
        table = pd.DataFrame()
    table.to_csv(os.path.join(filename+'.csv'), index=False)
    return table
#for file in jsonfilelist:
#    tmpdef('StreetFighter', file)

## these are the helper functions used consistently

In [ ]:
def generatevideodatalist(resp):
    allvideos = {}
    for i in resp:
        print(resp[i])
        items = resp[i]["items"]
        #print(items)
        for i in items:
            allvideos = allvideos.append(i)
    return allvideos

In [ ]:
def generatevideoidlistforkeyword(name, path, keyword, amount, orderedby, lang):
    os.chdir(WORKDIR)
    print(path)
    kw = name+'_'+keyword.replace('#', 'hashtag')+'_n'+str(amount)+'_'+orderedby
    data = getsearchresultsbykeyword(keyword, amount, orderedby, lang)
    if data:
        try:
            complist = pd.read_csv(os.path.join(path, f'{name}AllIdTable.csv'))
        except:
            print('creating new all id list')
            complist = pd.DataFrame()
        #print(complist.head())
        table = pd.DataFrame(columns = ['videoId', 'publishedAt', 'channelId', 'title',
                                        'description', 'channelTitle', 'publishTime', 'previousappearances'])
        for key in data:
            print(key)
            print(data[key]['id']['videoId'])
            #matches = complist[complist.eq(data[key]['id']['videoId']).any(axis=1)]
            matches = complist[complist.isin([data[key]['id']['videoId']])].stack()
            matches.index = matches.index.droplevel()
            matches = [x for x in matches.index]
            print(matches)
            tmp = {'videoId': [data[key]['id']['videoId']],
                   'publishedAt': [data[key]['snippet']['publishedAt']],
                   'channelId': [data[key]['snippet']['channelId']],
                   'title': [data[key]['snippet']['title']],
                   'description': [data[key]['snippet']['description']],
                   'channelTitle': [data[key]['snippet']['channelTitle']],
                   'publishTime': [data[key]['snippet']['publishTime']],
                   'previousappearances': str(matches)}
            table = pd.concat([table, pd.DataFrame.from_dict(tmp)], ignore_index = True)
    else:
        table = pd.DataFrame()
    table.to_csv(os.path.join(path, f'{kw}_videolist_{str(date.today())}.json.csv'), index=False)
    with open(os.path.join(path, f'{kw}_videolist_{str(date.today())}.json'), 'w+') as file:
        json.dump(data, file)
    return [data[item]['id']['videoId'] for item in data if 'videoId' in data[item]['id'].keys()]

In [ ]:
def createvideoidlists (name, path, searchconditionsdict):
    keyitemlist = {}
    for item in searchconditionsdict:
        print(item)
        print(searchconditionsdict[item]['searchterm'])
        print(searchconditionsdict[item]['language'])
        print(searchconditionsdict[item]['count'])
        print(searchconditionsdict[item]['order'])
        keyitemlist[item+str(date.today())] = generatevideoidlistforkeyword(name, path, searchconditionsdict[item]['searchterm'],
                                                                            searchconditionsdict[item]['count'], searchconditionsdict[item]['order'], 
                                                                            searchconditionsdict[item]['language'])
    return keyitemlist

# data evaluation and enhancement

In [ ]:
def findalljsonfilesindir(dir):
    os.chdir(WORKDIR)
#    print(os.listdir())
    result = []
    for file in os.listdir(dir):
        if file.endswith('.json') and not file.startswith('client'):
    #        print(os.path.join(dir, file))
            result.append(os.path.join(dir, file))
    return result

In [ ]:
def addliststocompletelist(name, path, lists):
    print(lists)
    try:
        previouslist = pd.read_csv(os.path.join(path, f'{name}AllIdTable.csv'), index_col=0)
        previouslist.to_csv(os.path.join(path, f'{name}AllIdTable{str(date.today())}.csv'), index=False)
        print(previouslist.shape)
    except:
        print('error: all id list does not exist')
        previouslist = pd.DataFrame()
    newlist = pd.DataFrame()
    newdict = {k: v for k, v in lists.items() if len(v) > 0}
    pprint.pprint(newdict)
    try:
        tmpdf = pd.DataFrame({k: pd.Series(v) for k, v in newdict.items()})
        print(tmpdf.shape)
        newlist = pd.concat([previouslist.reset_index(drop=True), tmpdf.reset_index(drop=True)], axis = 1)
        print(newlist.shape)
        newlist.head()
        newlist.to_csv(os.path.join(path, f'{name}AllIdTable.csv'), index = False)
    except:
        print('lists not compatible')
        print(previouslist.shape)
    return newlist
    #newlist.to_csv(os.path.join(paths[name], f'{name}AllIdTable.csv'))

In [ ]:
#print(addliststocompletelist(tmplist))

# execution section

In [ ]:
# Load outsourced config mapping
with open("config/youtube_search_configs.json", "r", encoding="utf-8") as f:
    master_search_config = json.load(f)
pprint.pprint(master_search_config)

In [ ]:
# Select which game dictionary you want to load from your config layout (e.g., 'Suika', 'Zelda', etc.)
TARGET_GAME = ""
viewcountonly = True
# Extract path string and search dict safely
try:
    game_path = master_search_config[TARGET_GAME]["paths"]
    search_conditions = master_search_config[TARGET_GAME]["searches"]
    print(f"Targeting Folder Path: {game_path}")

    if viewcountonly:
        # Filter search conditions to only include those with 'order' set to 'viewCount'
        search_conditions = {k: v for k, v in search_conditions.items() if key.endswith("view")}

    # Optional: Print out the loaded configuration to verify it matches
    print(f"Loaded config parameters for: {TARGET_GAME}")
    pprint.pprint(search_conditions)

except KeyError as e:
    print(f"Error: {e}")
    print('data not available')

if game_path and search_conditions:
    # Execute verification and retrieval loop using the configured path
    results = createvideoidlists(TARGET_GAME, game_path, search_conditions)
    with open (os.path.join(game_path, f'{name}_videoidlists_{str(date.today())}.yml'), 'w+') as file:
    for list in results:
        file.write(list+':\n')
        for item in results[list]:
            file.write(f" - '{item}'\n")
            
    # Build out and reconcile tracking sheets in the correct folder
    addliststocompletelist(TARGET_GAME, game_path, results)